In [1]:
"""
IMPORT ALL IMPORTS, DEPENDENCIES, ETC NEEDED
"""
from huggingface_hub import login
from datasets import load_dataset
import unicodedata
import pandas as pd
import torch
from transformers import (AutoTokenizer,AutoModelForSeq2SeqLM,Trainer,TrainingArguments,)
from torch.utils.data import DataLoader
import editdistance

/home/jay/anaconda3/envs/seq2seqyorubaadr/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#DIACRITIC STRIPPING FUNCTION
#FOR CREATING DIACRITIC-FREE INPUT ENTRIES
def strip_diacritics(text):
    return ''.join(
        #USES UNICODE TO STANDARDIZE, SUCH AS ọ TO o
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
        )

#FUNCTION TO PREP INPUT
#GIVEN ENTRY, MAKES DIACRITIC-FREE INPUT, DIACRITIC MARKED TARGET
def preprocess_hf(example):
    return {
        "input_text": strip_diacritics(example["text"]),
        "target_text": example["text"]
    }

#TOKENIZATION FUNCTION
def preprocess(example):
    #TOKENIZE INPUT
    model_inputs = tokenizer(
        example["input_text"],
        truncation=True,
        #PADS SHORT ENTRIES TO 256
        padding="max_length",
        #CAPS LONG ENTRIES TO 256
        #WE MIGHT WANT TO EXTEND THIS, OR IMPLEMENT SLIDING WINDOW, SINCE THIS LOSES INPUT INFORMATION
        max_length=256,
    )
    #TOKENIZE TARGET, SAME SETUP
    labels = tokenizer(
        example["target_text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )["input_ids"]

    #PADDING TOKENS
    #LIST COMPREHENSION FOR ALL LABEL IDS, WITH ALL PADDING TOKENS SET TO -100
    #CROSS ENTROPY LOSS IGNORES VALUE -100
    labels = [(l if l != tokenizer.pad_token_id else -100) for l in labels]
    model_inputs["labels"] = labels
    return model_inputs

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
import torch

model_path = "byt5_yoruba_75"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,  # matches your bf16 training
)

# important: disable gradient checkpointing for eval
model.gradient_checkpointing_disable()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()
model.to(device)
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 172/172 [00:01<00:00, 171.30it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/bin/nvidia-modprobe: unrecognized option: "-s"

ERROR: Invalid commandline, please run `/usr

T5ForConditionalGeneration(
  (shared): Embedding(384, 1472)
  (encoder): T5Stack(
    (embed_tokens): Embedding(384, 1472)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1472, out_features=384, bias=False)
              (k): Linear(in_features=1472, out_features=384, bias=False)
              (v): Linear(in_features=1472, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=1472, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1472, out_features=3584, bias=False)
              (wi_1): Linear(in_features=1472, out_features=3584, bias=False)
              (w

In [4]:
"""
EVALUATION
SWITCH MODEL TO EVAL MODE
"""

#SET TO DEVICE, SWITCH TO EVAL MODE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

"""
DEFINE EVAL HELPER FUNCTIONS

FUNCTION TO TRANSCODE ID VALUES BACK TO ACTUAL TEXT, FOR METRICS
FUNCTION TO CALCULATE CHARACTER ACCURACY
FUNCTION TO CALCULATE CHARACTER ERROR RATE

CAN ADD FURTHER EVAL METRIC FUNCTIONS HERE
"""

#FUNCTION TO DECODE IDS BACK TO REAL TEXT
def decode(ids):
    #SKIPS SPECIAL TOKENS SO STUFF LIKE [CLS] DOESNT GET WRITTEN INTO TEXT
    return tokenizer.decode(ids, skip_special_tokens=True)

#FUNCTION FOR CHARACTER ACCURACY METRIC
def char_accuracy(preds, targets):
    total = 0
    correct = 0
    #FOR PREDICTION, TARGET
    for p, t in zip(preds, targets):
        #GET SMALLER LEN
        min_len = min(len(p), len(t))
        correct += sum(p[i] == t[i] for i in range(min_len))
        total += len(t)
    #RETURN CHARACTER ACCURACY VALUE
    return correct / total if total > 0 else 0

#FUNCTION FOR CHARACTER ERROR RATE METRIC
def cer(preds, targets):
    total_dist = 0
    total_chars = 0
    #FOR PREDICTION, TARGET
    for p, t in zip(preds, targets):
        #GET ALL EDIT DISTANCES
        total_dist += editdistance.eval(p, t)
        #GET TOTAL CHAR LENGTH
        total_chars += len(t)
    #RETURN CER VALUE
    return total_dist / total_chars if total_chars > 0 else 0



In [5]:
import json
from datasets import Dataset

yad_data = []
with open("yad_test.json", "r", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        yad_data.append({
            "input_text": entry["translation"]["unyo"],
            "target_text": entry["translation"]["dcyo"]
        })

yad_dataset = Dataset.from_list(yad_data)
tokenized_yad = yad_dataset.map(preprocess, load_from_cache_file=False)

Map: 100%|██████████| 3330/3330 [01:02<00:00, 52.92 examples/s]


In [7]:
tokenized_yad.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [8]:
print(tokenized_yad)

Dataset({
    features: ['input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3330
})


In [9]:
from sacrebleu.metrics import BLEU

def bleu_score(preds, targets):
    bleu = BLEU()
    result = bleu.corpus_score(preds, [targets])
    return result.score

In [10]:
#EVAL DATALOADER
eval_loader = DataLoader(
    tokenized_yad,
    batch_size=16
)


"""
RUN MODEL IN EVAL MODE

ENDS WITH PREDICTED AND TARGETS ENTRIES, AS REAL TEXT
"""

#INSTANSIATE PREDICTION, TARGET LISTS
all_preds = []
all_targets = []

#RUN EVAL PASS THROUGH MODEL
with torch.no_grad():
    #PER BATCH
    for batch in eval_loader:
        #INPUTS GO IN
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        #GET PREDICTIONS
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=256
        )
        #CONVERT PREDICTIONS BACK TO REAL TEXT
        preds = [decode(g) for g in generated]
        #GET TARGETS
        targets = [
            #CONVER TARGETS BACK TO REAL TEXT
            decode([t.item() for t in label if t != -100])
            for label in batch["labels"]
        ]
        #ADD PREDICTIONS AND TARGETS (REAL TEXT) TO LISTS
        all_preds.extend(preds)
        all_targets.extend(targets)

"""
CALCULATE FINAL METRICS FOR EVAL SPLIT, PRINT
"""

#PRINT METRICS
acc = char_accuracy(all_preds, all_targets)
cer_score = cer(all_preds, all_targets)
bleu = bleu_score(all_preds, all_targets)
print(f"\nCharacter Accuracy: {acc:.4f}")
print(f"CER: {cer_score:.4f}")
print(f"BLEU Score: {bleu:.2f}")

""""
PRINT SAMPLE SELECTION OF INPUT, TARGET, PREDICTION TRIOS

JUST FOR EYEBALLING HOW SYSTEM IS DOING
"""

#PRINT SOME ACGTUAL SAMPLE METRICS
print("\nSample Predictions:\n" + "-"*50)
for i in range(min(10, len(yad_dataset))):
    print(f"Input     : {yad_dataset[i]['input_text']}")
    print(f"Target    : {all_targets[i]}")
    print(f"Prediction: {all_preds[i]}")
    print("-"*50)



Character Accuracy: 0.3704
CER: 0.2213
BLEU Score: 32.29

Sample Predictions:
--------------------------------------------------
Input     : Ile- ise olopaa ni ipinle  Oyo ti se afihan afurasi odaran kan, Abegunde Olaniyi to lowo ninu iku Abileko Grace Ajibola ti se ajihinrere ni ile ijosin omoleyin Kristi kan ni agbegbe  Oluyole ni ilu Ibadan to wa ni ekun Gusu iwo oorun orile ede Naijiria ni ojo ketadinlogun osu keta odun 2020 yii.
Target    : Ilé- iṣẹ ọlọ́pàá ní ìpínlẹ̀  Ọ̀yọ́ ti se àfihàn afurasí ọ̀daràn kan, Abégúndé Oláníyì tó lọ́wọ́ nínú ikú Abilékọ Grace Ajíbọ́lá tí ṣe ajíhìnrere ní ilé ìjọsìn ọmọlẹ́yìn Kristi kan n
Prediction: Ilé- iṣẹ́ ọlọ́pàá ní ìpínlẹ̀  Oyo ti ṣe àfihàn afurasí ọ̀daràn kan, Abegunde Olaniyi tó lọ́wọ́ nínú ikú Abileko Grace Ajibola ti ṣe ajihinrere ni ilé ìjọsìn ọmọlẹ́yìn Kristi kan ní agbègbè  Oluyole n
--------------------------------------------------
Input     : Aregbesola wa ro gbogbo awon omoleyin kristi lati maa sawokose iwa ati abuda Jesu kristi nip